# Assignment 2: Image Processing and Pyramid Blending

**Individual · 100 points**

This assignment investigates how filtering, dynamic-range transformations, multi-scale representations, and blending improve visual data.

## 0. Student Information

- First name: **REPLACE ME**
- Last name: **REPLACE ME**
- ABC123: **REPLACE ME**
- External resources and generative-AI assistance: **REPLACE ME with each meaningful resource/use, or None**

This is an individual assignment. Do not install unapproved packages or use external libraries or pretrained pipelines to bypass a required implementation.

Generative AI tools may be used for conceptual clarification, debugging assistance, code suggestions, and help interpreting documentation. Students must disclose meaningful use in every assignment. You remain responsible for understanding, testing, and explaining all submitted work. AI may not replace your own experimental analysis, failure analysis, reflection, or interpretation of observed results. You may be asked to explain selected work; inability to do so may reduce credit or lead to an academic-integrity review.


## 1. Assignment Overview and Learning Objectives

By completing A2, you should be able to implement and analyze spatial and nonlinear filtering; compare denoising under distinct noise models; manipulate contrast and dynamic range; construct and reconstruct Gaussian and Laplacian pyramids; perform Laplacian pyramid blending; evaluate results with visual and quantitative evidence; and diagnose limitations.

**Point map:** core implementation 40; required experiments and evidence 21; pyramid decomposition 6; analysis and interpretation 18; reproducibility, organization, and submission compliance 10; extension 5.

| Tables & Metadata | Figures | Other Outputs |
|---|---|---|
| `experiment_metrics.csv` | `filtering_results.png` | one declared extension artifact |
| `pyramid_metrics.json` | `denoising_comparison.png` | |
| | `contrast_enhancement.png` | |
| | `pyramid_visualization.png` | |
| | `pyramid_reconstruction.png` | |
| | `blending_comparison.png` | |
| | `pyramid_decomposition.png` | |
| | `failure_analysis.png` | |


## 2. Environment Verification

Run this setup first. It discovers the repository independently of the notebook working directory and verifies the immutable assets.


In [ ]:
from pathlib import Path
import json, sys
import cv2
import imageio.v3 as iio
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from skimage.metrics import mean_squared_error, peak_signal_noise_ratio, structural_similarity
import cs5243
from cs5243.data import find_course_root
COURSE_ROOT=find_course_root(Path(cs5243.__file__))
A2=COURSE_ROOT/"A2"; sys.path.insert(0,str(A2/"src"))
import a2_tools
import student_code as sc
PATHS=a2_tools.ensure_output_directories()
problems=a2_tools.verify_asset_hashes(); assert not problems, problems
print({"assignment":"A2-2026.1","environment":"2026.1"})


## 3. Dataset and Problem Setup

The dataset contains odd dimensions, strong edges, gradients, thin structure, texture, controlled Gaussian and impulse noise, five aligned blend pairs with different mask geometry (a soft vertical seam, a circular spot, and a diagonal seam over synthetic scenes, plus a hard-edged vertical seam and a hard-edged circular seam over real fruit photographs), an underexposed contrast source, and a high-frequency difficult case. Most assets are generated programmatically (CC0 1.0); the two real-photo blend pairs are retained course-legacy teaching images credited in `data/ATTRIBUTION.md`. Unlike the synthetic masks, the real-photo masks are hard-edged rather than feathered, so direct blending (splicing) leaves an obvious seam that Laplacian pyramid blending visibly smooths. Do not edit the assets.

Implement reusable graded functions in `src/student_code.py`; use the notebook to call them, conduct experiments, save required artifacts, and explain results. Maintain one cumulative `outputs/tables/experiment_metrics.csv` throughout the notebook using the documented column names, and append later finite-valued records without deleting earlier experiments.

In [ ]:
images=a2_tools.load_images()
print({name:(value.shape,str(value.dtype)) for name,value in images.items()})


## 4. Core Implementation

### Task 1 — convolution, Gaussian kernels, and sharpening (10 implementation + 6 evidence points)

Implement `convolve2d_manual`, `gaussian_kernel`, and `sharpen_image` to their contracts. `convolve2d_manual` is true convolution, including spatial kernel reversal; it must not call a library convolution/correlation routine. Compare your filtering with an explicitly identified library reference only in the experiment. Preserve numeric values until display or saving. Save `outputs/figures/filtering_results.png` and append filtering/reference-error evidence to `experiment_metrics.csv`.


In [ ]:
raise NotImplementedError("Complete Task 1 and its controlled filtering experiment")


**Filtering analysis (3–5 sentences):** Connect kernel support and sigma to smoothing, edge behavior, detail loss, halo formation, and measured evidence.


### Task 2 — noise models and denoising (8 implementation + 6 evidence points)

Implement deterministic Gaussian and impulse noise plus `median_filter_manual`. With declared seeds, use your two noise functions to create the observations used in the comparison. The distributed `gaussian_noisy.png` and `impulse_noisy.png` files are independent observations drawn with the same noise parameters; use them for visual comparison only. They are not bit-identical to your own output, so do not spend time trying to reproduce them exactly. Compare Gaussian, median, and OpenCV bilateral filtering on both noise types, using the same clean reference. Save `outputs/figures/denoising_comparison.png` and append MSE or RMSE, PSNR, and SSIM records to `experiment_metrics.csv`. Metrics supplement rather than replace visual evidence.


In [ ]:
raise NotImplementedError("Complete Task 2 and its denoising experiment")


**Denoising analysis (3–5 sentences):** Explain why rankings differ by noise model and identify a detail-preservation tradeoff.


### Task 3 — contrast and dynamic range (7 implementation + 5 evidence points)

Implement percentile normalization, manual grayscale histogram equalization, and gamma correction. Compare at least three methods or settings on a controlled source. Address clipping, tonal distribution, noise amplification, and luminance-versus-channel processing. Save `outputs/figures/contrast_enhancement.png` and append measurements to the cumulative metrics file. OpenCV CLAHE may be a comparison but not a substitute for required work.


In [ ]:
raise NotImplementedError("Complete Task 3 and its contrast experiment")


**Contrast analysis (3–5 sentences):** Tie visible changes to clipped fractions or tonal measurements and state one limitation.


### Task 4 — Gaussian and Laplacian pyramids (8 implementation + 4 evidence points)

Implement all three pyramid functions with finest-to-coarsest ordering, multi-channel support, and correct reconstruction for odd dimensions. Approved resizing operations are permitted; pyramid logic remains student work. Save `pyramid_visualization.png`, `pyramid_reconstruction.png`, and `pyramid_metrics.json` containing the required reconstruction fields documented in the README.


In [ ]:
raise NotImplementedError("Complete Task 4 and its reconstruction experiment")


**Pyramid analysis (3–5 sentences):** Explain what different levels represent and use reconstruction error to distinguish mathematical correctness from visual plausibility.


## 5. Required Experiments

### Task 5 — direct blending and Laplacian pyramid blending across multiple pairs (7 implementation + 3 evidence points)

Implement `laplacian_pyramid_blend`, where mask 0 selects `image_a`, mask 1 selects `image_b`, and intermediate values blend. The dataset provides five aligned blend pairs with different mask geometry: `blend_left`/`blend_right` with `blend_mask` (soft vertical seam) or `blend_mask_spot` (circular seam), `blend2_left`/`blend2_right` with `blend2_mask` (diagonal seam), `blend3_left`/`blend3_right` (real apple/orange photographs) with `blend3_mask` (hard vertical seam), and `blend4_left`/`blend4_right` (a second real apple/orange close-up pair) with `blend4_mask` (hard circular seam). The two real-photo masks are not feathered, so they give the clearest evidence of the difference between the two methods. Apply `laplacian_pyramid_blend` to at least two of these five pairs, comparing each against direct blending (splicing) using the same mask, and include one deliberately poor mask or configuration (for example, a hard-thresholded mask or too few pyramid levels) on at least one pair. Save one combined `outputs/figures/blending_comparison.png` arranging results across the pairs you used, and append per-pair seam or transition evidence to the cumulative metrics file.

In [ ]:
raise NotImplementedError("Complete Task 5 and its blending experiment across multiple pairs")

**Blending analysis (3–5 sentences):** Discuss seam visibility, low- and high-frequency transitions, mask design, and boundary artifacts.


### Task 6 — pyramid decomposition visualization (4 evidence + 2 analysis points)

For one synthetic pair and one real-photo pair (state which you used), use your already-implemented `gaussian_pyramid`, `laplacian_pyramid`, and `laplacian_pyramid_blend` to make the blending process itself visible instead of only its result. For each pair, show: the Gaussian pyramid of each source image; the Laplacian pyramid of each source image, rescaled per level so the band-pass structure is visible instead of near-black; the blended Laplacian pyramid produced by combining each level with the mask's Gaussian pyramid; and the progressive reconstruction from the coarsest level up to the final full-resolution blend. Save one combined `outputs/figures/pyramid_decomposition.png` arranging both pairs. This task exercises functions you already implemented in Tasks 4 and 5 — no new graded function is required.

In [ ]:
raise NotImplementedError("Complete Task 6's pyramid decomposition visualization")

**Decomposition analysis (2–4 sentences):** Explain what a Laplacian pyramid level represents, why blending per level instead of blending final pixel values directly avoids a visible seam, and connect this to the reconstruction check from Task 4.

## 6. Results and Analysis

The immediate experiment responses contribute **12 analysis points collectively**. The synthesis contributes **3 points**. In at most 180 words, connect the most important filtering tradeoff, the strongest multi-scale finding, and one limitation. Do not restate each earlier observation.


**Synthesis:** YOUR RESPONSE.


## 7. Failure Analysis

### Task 7 — two controlled failures (5 points)

Show two reproducible failures from different topics. For each, identify the failure, show evidence, explain its mechanism, correct or mitigate it, and verify the improvement. The approved fallback is an excessively large sharpening amount that produces halos around strong edges, followed by a reduced or reformulated configuration. Students may instead use another reproducible failure. Save one coherent `outputs/figures/failure_analysis.png` and append `failure` rows to `experiment_metrics.csv` that quantify each failure and its mitigation.


In [ ]:
raise NotImplementedError("Create and verify two failure analyses")


**Failure analysis:** YOUR RESPONSE.


## 8. Extension

### Task 8 — Extension (5 points)

Choose one: compare an approved advanced denoiser; alternative-color-space sharpening; local contrast enhancement; frequency-domain filtering; hybrid image; pyramid-mask comparison; small focus stack; or an approved enhancement. Submit one declared principal artifact under `outputs/extension/`, include one baseline comparison, and give a concise evidence-based explanation. Use no additional packages or pretrained models.


In [ ]:
raise NotImplementedError("Complete one extension")


**Extension question and conclusion (80–130 words total):** YOUR RESPONSE.


## 9. Reflection and Reproducibility

Reproducibility, organization, and submission compliance are worth **10 points**: 4 for reproducible decisions and reflection, and 6 for correct execution, artifacts, schemas, disclosure, and packaging. In 100–150 words, identify one implementation decision another person must reproduce, one metric limitation, and one practice you will carry forward. Update the disclosure in Section 0.


**Reflection:** YOUR RESPONSE.


## 10. Submission Validation

Restart the kernel, run all cells, and export current HTML as `A2.html`. From any terminal directory, use the repository path:

```bash
python /path/to/common-setup/scripts/validate_submission.py --assignment A2
python /path/to/common-setup/scripts/package_submission.py --assignment A2
```

Submit the ASCII-safe `LastName_FirstName_A2.zip`. Do not submit PDF, datasets, caches, environments, checkpoints, or nested ZIP files.


### Canvas submission checklist

- [ ] The ZIP contains exactly one top-level folder named `LastName_FirstName_A2`.
- [ ] Section 0 identity fields (first name, last name, ABC123) are complete.
- [ ] `A2.ipynb` was restarted, run top to bottom, and saved without errors, and `A2.html` was exported after that final run.
- [ ] `src/` contains the required source files: `a2_tools.py` and `student_code.py`.
- [ ] `outputs/` contains only the required figures, tables, the fused image, and one extension artifact — no extra files.
- [ ] `python scripts/validate_submission.py --assignment A2` reports zero errors; review any warnings.
- [ ] No datasets, environments, caches, checkpoints, hidden files, or temporary files are included.
- [ ] External resources and meaningful generative-AI use are disclosed in Section 0.
- [ ] A PDF export is not required and is not included.


In [ ]:
for category in ("figures","tables","images","extension"): print(category,sorted(p.name for p in PATHS[category].glob("*") if p.is_file()))
